# Alzheimer's Diagnosis Analysis with Random Forest

This notebook is designed to do more than just train a model. It helps answer questions such as:

- How balanced is the dataset?
- Which variables differ between diagnosis groups?
- Which features dominate prediction?
- How much performance depends on cognitive/functional variables?
- Are non-cognitive risk factors useful at all?
- What can be written in the discussion section based on the results?

**How to use it**
1. Put your CSV file in the same folder as this notebook.
2. Update `DATA_PATH` below if needed.
3. Run the notebook top to bottom.
4. Save the outputs and we can interpret them together.

In [ ]:
# =========================
# 1. Imports
# =========================
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub as kh
import os

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc
)
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [ ]:
path = kh.dataset_download("rabieelkharoua/alzheimers-disease-dataset")
print("Path to dataset files:", path)

files = os.listdir(path)
print("Files in directory:", files)

csv_file = [f for f in files if f.endswith('.csv')][0]
csv_path = os.path.join(path, csv_file)


df = pd.read_csv(csv_path)
print("Shape:", df.shape)
df.head()

## 3. Basic structure and missing values

This section checks:
- column types
- missing values
- whether `PatientID` or `DoctorInCharge` should be excluded from modeling

In [ ]:
print(df.info())
print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

In [ ]:
# Drop obvious non-predictive identifiers if present
drop_if_present = ["PatientID", "DoctorInCharge"]
df = df.drop(columns=[c for c in drop_if_present if c in df.columns])

print("Columns after dropping ID-like columns:")
print(df.columns.tolist())

## 4. Target distribution

This helps judge whether accuracy is reliable or inflated by class imbalance.

In [ ]:
target_col = "Diagnosis"

print(df[target_col].value_counts(dropna=False))
print("\nClass proportions:")
print(df[target_col].value_counts(normalize=True).round(3))

plt.figure(figsize=(6,4))
sns.countplot(data=df, x=target_col)
plt.title("Diagnosis Distribution")
plt.show()

## 5. Split columns by type

This notebook treats text/category columns differently from numeric columns.

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = [c for c in df.columns if c not in categorical_cols + [target_col]]

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

## 6. Quick descriptive statistics by diagnosis

This section compares group means for numeric variables.  
It does **not** prove causality, but it helps spot variables worth discussing.

In [ ]:
group_summary = df.groupby(target_col)[numeric_cols].agg(["mean", "median", "std"]).round(2)
display(group_summary)

## 7. Univariate visual analysis

These plots help answer:
- Do diagnosed and non-diagnosed groups look different?
- Are the strongest variables cognitive/functional?
- Are there visible differences in age, sleep, activity, BMI, cholesterol, etc.?

You can add or remove features in `features_to_plot`.

In [ ]:
features_to_plot = [
    c for c in [
        "Age", "BMI", "PhysicalActivity", "DietQuality", "SleepQuality",
        "SystolicBP", "DiastolicBP", "CholesterolTotal", "CholesterolLDL",
        "CholesterolHDL", "CholesterolTriglycerides",
        "MMSE", "FunctionalAssessment", "ADL", "MemoryComplaints",
        "BehavioralProblems"
    ] if c in df.columns
]

n_cols = 3
n_rows = int(np.ceil(len(features_to_plot) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, features_to_plot):
    sns.boxplot(data=df, x=target_col, y=col, ax=ax)
    ax.set_title(f"{col} by Diagnosis")

for ax in axes[len(features_to_plot):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 8. Correlation with diagnosis (numeric only)

This is useful for a quick ranking of which numeric variables are most associated with the target.  
Because the target is binary, correlation is only a rough guide, not a full explanation.

In [ ]:
corr_df = df[numeric_cols + [target_col]].corr(numeric_only=True)
corr_with_target = corr_df[target_col].drop(target_col).sort_values(key=lambda s: s.abs(), ascending=False)

plt.figure(figsize=(8, max(6, len(corr_with_target)*0.35)))
sns.heatmap(corr_with_target.to_frame(), annot=True, cmap="coolwarm", center=0)
plt.title("Correlation of Numeric Features with Diagnosis")
plt.show()

display(corr_with_target.to_frame("correlation_with_diagnosis"))

## 9. Modeling helper functions

We build two model settings:

1. **Full model**: includes everything except ID-like columns  
2. **Reduced model**: removes variables that are very close to diagnosis itself

The goal is to compare:
- diagnostic performance
- risk-factor performance

In [ ]:
def build_preprocessor(X):
    cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )
    return preprocessor


def evaluate_model(X, y, model_name="Model", threshold=0.5, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )

    preprocessor = build_preprocessor(X)

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=random_state,
            class_weight="balanced",
            oob_score=True,
            n_jobs=-1
        ))
    ])

    pipeline.fit(X_train, y_train)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    results = {
        "model_name": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }

    # Extract OOB score (only available after fitting, not via pipeline directly)
    rf_model = pipeline.named_steps["model"]
    oob = rf_model.oob_score_
    results["oob_score"] = oob

    print(f"===== {model_name} =====")
    print("Metrics:")
    for k, v in results.items():
        if k != "model_name":
            print(f"{k}: {v:.3f}")

    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap="Blues")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    pr, rc, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(rc, pr)

    plt.figure(figsize=(6,4))
    plt.plot(fpr, tpr, label=f"AUC = {results['roc_auc']:.3f}")
    plt.plot([0,1], [0,1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {model_name}")
    plt.legend()
    plt.show()

    plt.figure(figsize=(6,4))
    plt.plot(rc, pr, label=f"PR AUC = {pr_auc:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve - {model_name}")
    plt.legend()
    plt.show()

    return pipeline, X_train, X_test, y_train, y_test, y_proba, y_pred, results

## 10. Full model

This includes all available features except dropped ID-like columns.

In [ ]:
X_full = df.drop(columns=[target_col])
y = df[target_col]

full_model, X_train_full, X_test_full, y_train_full, y_test_full, y_proba_full, y_pred_full, full_results = evaluate_model(
    X_full, y, model_name="Full Random Forest"
)

## 11. Permutation importance for full model

Permutation importance is better than raw tree importance for interpretation.  
If cognitive or functional variables dominate here, that is strong evidence that the model is largely diagnostic.

In [ ]:
perm_full = permutation_importance(
    full_model, X_test_full, y_test_full,
    n_repeats=10, random_state=42, n_jobs=-1
)

importance_full = pd.DataFrame({
    "feature": X_test_full.columns,
    "importance_mean": perm_full.importances_mean,
    "importance_std": perm_full.importances_std
}).sort_values("importance_mean", ascending=False)

display(importance_full.head(20))

plt.figure(figsize=(10,6))
sns.barplot(data=importance_full.head(15), x="importance_mean", y="feature")
plt.title("Top 15 Permutation Importances - Full Model")
plt.xlabel("Mean decrease in score")
plt.ylabel("Feature")
plt.show()

## 12. Reduced model without near-diagnostic cognitive/functional variables

These columns are often too close to the target itself.  
Adjust the list if your column names differ.

In [ ]:
near_diagnostic_cols = [
    c for c in [
        "MMSE",
        "FunctionalAssessment",
        "ADL",
        "MemoryComplaints",
        "BehavioralProblems",
        "Confusion",
        "Disorientation",
        "PersonalityChanges",
        "DifficultyCompletingTasks",
        "Forgetfulness",
    ] if c in df.columns
]

print("Removed from reduced model:", near_diagnostic_cols)

X_reduced = df.drop(columns=[target_col] + near_diagnostic_cols)

reduced_model, X_train_red, X_test_red, y_train_red, y_test_red, y_proba_red, y_pred_red, reduced_results = evaluate_model(
    X_reduced, y, model_name="Reduced Random Forest (risk factors only)"
)

## 13. Permutation importance for reduced model

This is one of the most interesting outputs in the notebook.  
Even if performance is weak, this can still show which non-cognitive factors matter most.

In [ ]:
perm_red = permutation_importance(
    reduced_model, X_test_red, y_test_red,
    n_repeats=10, random_state=42, n_jobs=-1
)

importance_red = pd.DataFrame({
    "feature": X_test_red.columns,
    "importance_mean": perm_red.importances_mean,
    "importance_std": perm_red.importances_std
}).sort_values("importance_mean", ascending=False)

display(importance_red.head(20))

plt.figure(figsize=(10,6))
sns.barplot(data=importance_red.head(15), x="importance_mean", y="feature")
plt.title("Top 15 Permutation Importances - Reduced Model")
plt.xlabel("Mean decrease in score")
plt.ylabel("Feature")
plt.show()

## 14. Compare the two model settings

This table often becomes one of the clearest figures in the report.

In [ ]:
comparison = pd.DataFrame([full_results, reduced_results]).set_index("model_name").round(3)
display(comparison)

## 15. Threshold tuning for the reduced model

In medical problems, recall for the positive class can matter more than accuracy.  
This section tests several thresholds to see whether we can catch more positive cases.

In [ ]:
thresholds = np.arange(0.2, 0.8, 0.05)
rows = []

for t in thresholds:
    y_pred_t = (y_proba_red >= t).astype(int)
    rows.append({
        "threshold": round(float(t), 2),
        "accuracy": accuracy_score(y_test_red, y_pred_t),
        "precision": precision_score(y_test_red, y_pred_t, zero_division=0),
        "recall": recall_score(y_test_red, y_pred_t, zero_division=0),
        "f1": f1_score(y_test_red, y_pred_t, zero_division=0)
    })

threshold_df = pd.DataFrame(rows)
display(threshold_df)

plt.figure(figsize=(8,5))
plt.plot(threshold_df["threshold"], threshold_df["precision"], marker="o", label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], marker="o", label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], marker="o", label="F1")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Tuning - Reduced Model")
plt.legend()
plt.show()

## 16. Cross-validated AUC

A single train/test split can be noisy.  
This section gives a more stable view of model quality.

In [ ]:
def cv_auc_score(X, y, random_state=42):
    preprocessor = build_preprocessor(X)
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=random_state,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    auc_scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    return auc_scores

full_cv_auc = cv_auc_score(X_full, y)
reduced_cv_auc = cv_auc_score(X_reduced, y)

print("Full model CV ROC-AUC:", np.round(full_cv_auc, 3), "| mean =", round(full_cv_auc.mean(), 3))
print("Reduced model CV ROC-AUC:", np.round(reduced_cv_auc, 3), "| mean =", round(reduced_cv_auc.mean(), 3))

## 17. Age subgroup analysis

This is useful because Alzheimer’s risk may behave differently in older subgroups.  
You can change the age cutoff if needed.

In [ ]:
if "Age" in df.columns:
    age_cutoff = 75
    older_df = df[df["Age"] >= age_cutoff].copy()

    print(f"Subset size for Age >= {age_cutoff}:", older_df.shape)
    if older_df.shape[0] > 100 and older_df[target_col].nunique() == 2:
        X_old = older_df.drop(columns=[target_col] + near_diagnostic_cols, errors="ignore")
        y_old = older_df[target_col]

        old_model, X_train_old, X_test_old, y_train_old, y_test_old, y_proba_old, y_pred_old, old_results = evaluate_model(
            X_old, y_old, model_name=f"Reduced Model in Age >= {age_cutoff}"
        )
        print(old_results)
    else:
        print("Not enough rows or not enough class variation for this subgroup.")
else:
    print("Age column not found.")

## 18. Quick binary-feature prevalence comparison

This is helpful for variables like:
- Smoking
- Diabetes
- Depression
- Hypertension
- FamilyHistoryAlzheimers

It shows the proportion of positive cases within each diagnosis group.

In [ ]:
binary_like_cols = []
for col in df.columns:
    if col != target_col and df[col].dropna().nunique() <= 2:
        binary_like_cols.append(col)

binary_like_cols = [c for c in binary_like_cols if c not in categorical_cols or df[c].dropna().isin([0,1]).all()]

if binary_like_cols:
    prevalence = df.groupby(target_col)[binary_like_cols].mean().T.sort_values(by=1 if 1 in df[target_col].unique() else df[target_col].unique()[-1], ascending=False)
    display(prevalence)

    plot_cols = prevalence.head(12).index.tolist()
    prevalence.loc[plot_cols].plot(kind="bar", figsize=(12,6))
    plt.title("Prevalence of Binary Features by Diagnosis Group")
    plt.ylabel("Proportion")
    plt.xticks(rotation=45, ha="right")
    plt.show()
else:
    print("No binary-like columns detected.")

## 19. Optional: export key result tables

Uncomment if you want CSV outputs for your report.

In [ ]:
# comparison.to_csv("model_comparison.csv")
# importance_full.to_csv("importance_full.csv", index=False)
# importance_red.to_csv("importance_reduced.csv", index=False)
# threshold_df.to_csv("threshold_tuning_reduced.csv", index=False)

## 20. Notes for writing the report

After running the notebook, look for answers to these:

1. Is the dataset strongly imbalanced?
2. Which variables differ most between diagnosis groups?
3. Does the full model rely mainly on cognitive/functional variables?
4. How much does performance drop in the reduced model?
5. Which non-cognitive variables remain important?
6. Does threshold tuning improve recall enough to be worth the precision loss?
7. Does subgroup analysis show stronger signal in older patients?

Once you have the outputs, we can turn them into:
- findings
- discussion points
- limitations
- conclusion text